In [ ]:
# ============================================================
# CELL 1 — IMPORT
# ============================================================

import pandas as pd
import numpy as np
import re
import shutil

from pathlib import Path
from datetime import datetime, date

from openpyxl import load_workbook

# Microsoft Excel COM
import win32com.client as win32

print("Libraries loaded successfully.")
print("Excel COM loaded successfully.")

: 

In [ ]:
# ============================================================
# CELL 2 — FILE PATH
# ============================================================

BASE_DIR = Path(
    r"C:\Users\zerox\OneDrive\Documents\Desktop\learning"
)

# ------------------------------------------------------------
# MASTER TEMPLATE
# ------------------------------------------------------------

MASTER_FILE = (
    BASE_DIR
    / "data"
    / "master"
    / "Learning agility self assesment & superior_24.7.2026.xlsx"
)

# ------------------------------------------------------------
# RAW ASSESSMENT
# ------------------------------------------------------------

SELF_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Self Assessment MASTER Questions Answers 1.xlsx"
)

SUPERIOR_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Superior Assessment MASTER Questions Answers.xlsx"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "master_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("========================================")
print("PATH CONFIGURATION")
print("========================================")

print("Master   :", MASTER_FILE)
print("Exists   :", MASTER_FILE.exists())

print("\nSelf     :", SELF_FILE)
print("Exists   :", SELF_FILE.exists())

print("\nSuperior :", SUPERIOR_FILE)
print("Exists   :", SUPERIOR_FILE.exists())

print("\nOutput   :", OUTPUT_DIR)
print("Exists   :", OUTPUT_DIR.exists())

In [ ]:
# ============================================================
# CELL 3 — LOAD MASTER + RAW
# ============================================================

assert MASTER_FILE.exists(), (
    f"Master file tidak ditemukan:\n{MASTER_FILE}"
)

assert SELF_FILE.exists(), (
    f"Self Assessment file tidak ditemukan:\n{SELF_FILE}"
)

assert SUPERIOR_FILE.exists(), (
    f"Superior Assessment file tidak ditemukan:\n{SUPERIOR_FILE}"
)

# ------------------------------------------------------------
# LOAD MASTER
# ------------------------------------------------------------

master_wb = load_workbook(
    MASTER_FILE,
    data_only=False
)

# ------------------------------------------------------------
# LOAD RAW SELF
# ------------------------------------------------------------

df_self = pd.read_excel(
    SELF_FILE,
    engine="openpyxl"
)

# ------------------------------------------------------------
# LOAD RAW SUPERIOR
# ------------------------------------------------------------

df_superior = pd.read_excel(
    SUPERIOR_FILE,
    engine="openpyxl"
)

print("========================================")
print("FILES LOADED")
print("========================================")

print("Self     :", df_self.shape)
print("Superior :", df_superior.shape)

print("\nMaster sheets:")

for sheet in master_wb.sheetnames:
    print("-", sheet)

In [ ]:
# ============================================================
# CELL 4 — TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    # Non-breaking space
    value = value.replace("\xa0", " ")

    # Line breaks
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    # Multiple spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()

In [ ]:
# ============================================================
# CELL 5 — EXTRACT MASTER 5 DIMENSIONS
# ============================================================

DIMENSIONS = {
    "mental_agility": "mental agility",
    "people_agility": "people agility",
    "change_agility": "change agility",
    "result_agility": "result agility",
    "self_awareness": "self awareness",
}

indicator_master = []

for dimension_key, sheet_name in DIMENSIONS.items():

    ws = master_wb[sheet_name]

    # Master memiliki 10 indikator:
    # Row 13 = indikator 1
    # Row 14 = indikator 2
    # ...
    # Row 22 = indikator 10

    for indicator_no, row in enumerate(range(13, 23), start=1):

        indicator = ws.cell(row, 2).value
        question = ws.cell(row, 3).value

        levels = {
            1: ws.cell(row, 4).value,
            2: ws.cell(row, 5).value,
            3: ws.cell(row, 6).value,
            4: ws.cell(row, 7).value,
            5: ws.cell(row, 8).value,
        }

        indicator_master.append({

            # ------------------------------------------------
            # IDENTITAS
            # ------------------------------------------------

            "dimension": dimension_key,

            "sheet_name": sheet_name,

            "indicator_no": indicator_no,

            "indicator_id": (
                f"{dimension_key}_{indicator_no:02d}"
            ),

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            "indicator": indicator,

            # ------------------------------------------------
            # QUESTION
            # ------------------------------------------------

            "question": question,

            "question_normalized": normalize_text(question),

            # ------------------------------------------------
            # LEVEL 1 - 5
            # ------------------------------------------------

            "level_1": levels[1],
            "level_2": levels[2],
            "level_3": levels[3],
            "level_4": levels[4],
            "level_5": levels[5],

            # ------------------------------------------------
            # NORMALIZED ANSWERS
            # ------------------------------------------------

            "level_1_normalized": normalize_text(levels[1]),
            "level_2_normalized": normalize_text(levels[2]),
            "level_3_normalized": normalize_text(levels[3]),
            "level_4_normalized": normalize_text(levels[4]),
            "level_5_normalized": normalize_text(levels[5]),
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

indicator_master_df = pd.DataFrame(indicator_master)


# ============================================================
# DISPLAY STRUCTURE
# ============================================================

print("========================================")
print("MASTER STRUCTURE")
print("========================================")

print(
    "Total indicators :",
    len(indicator_master_df)
)

print("\nIndicators per dimension:")

print(
    indicator_master_df
    .groupby("dimension")
    .size()
)

print("\nIndicator IDs:")

print(
    indicator_master_df[
        [
            "dimension",
            "indicator_no",
            "indicator_id",
            "indicator"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# CELL 6 — VALIDATE MASTER
# ============================================================

assert len(indicator_master_df) == 50, (
    f"Expected 50 indicators, "
    f"found {len(indicator_master_df)}"
)

assert (
    indicator_master_df["indicator_id"]
    .nunique()
    == 50
), "Duplicate indicator_id detected."

assert (
    indicator_master_df["question_normalized"]
    .nunique()
    == 50
), "Duplicate questions detected."

for col in [
    "level_1",
    "level_2",
    "level_3",
    "level_4",
    "level_5"
]:

    missing = (
        indicator_master_df[col]
        .isna()
        .sum()
    )

    assert missing == 0, (
        f"{col} has {missing} missing values."
    )


print("========================================")
print("MASTER VALIDATION")
print("========================================")
print("50 indicators       : OK")
print("50 unique questions : OK")
print("Level 1-5 complete  : OK")

In [ ]:
# ============================================================
# CELL 7 — DETECT RAW QUESTION COLUMNS
# ============================================================

def detect_question_columns(df, master_questions):
    master_question_set = {
        normalize_text(q) for q in master_questions if pd.notna(q)
    }

    question_columns = []
    for col in df.columns:
        normalized_col = normalize_text(col)
        if normalized_col in master_question_set:
            question_columns.append(col)

    return question_columns


master_questions = indicator_master_df["question"].tolist()

self_question_cols = detect_question_columns(df_self, master_questions)
superior_question_cols = detect_question_columns(df_superior, master_questions)

print("========================================")
print("RAW QUESTION VALIDATION")
print("========================================")

print("Self questions detected     :", len(self_question_cols))
print("Superior questions detected :", len(superior_question_cols))

assert len(self_question_cols) == 50, (
    f"Self raw contains {len(self_question_cols)} questions, expected 50 Master questions."
)

assert len(superior_question_cols) == 50, (
    f"Superior raw contains {len(superior_question_cols)} questions, expected 50 Master questions."
)

print("\nQuestion structure: OK")

In [ ]:
# ============================================================
# CELL 8 — MASTER ANSWER → RATING MAPPING
# ============================================================

answer_mapping = {}

for _, row in indicator_master_df.iterrows():

    question_key = row["question_normalized"]

    answer_mapping[question_key] = {

        normalize_text(row["level_1"]): 1,

        normalize_text(row["level_2"]): 2,

        normalize_text(row["level_3"]): 3,

        normalize_text(row["level_4"]): 4,

        normalize_text(row["level_5"]): 5,
    }


print("========================================")
print("ANSWER MAPPING")
print("========================================")

print(
    "Questions mapped :",
    len(answer_mapping)
)

print(
    "Answers mapped   :",
    sum(
        len(v)
        for v in answer_mapping.values()
    )
)

assert len(answer_mapping) == 50

print("50 questions × 5 ratings = 250 mappings")

In [ ]:
# ============================================================
# CELL 9 — RAW → SCORED DATA FUNCTION
# ============================================================

def clean_value(value):
    """Membersihkan value dari Excel. NaN / None → string kosong."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def process_assessment(df, question_columns, rater_type):
    results = []

    for row_index, row in df.iterrows():
        # Clean helper for row dict
        row_dict = {normalize_text(k): v for k, v in row.items()}

        if rater_type == "Self":
            employee_name = clean_value(row.get("Nama", row_dict.get("nama", "")))
            superior_name = ""
        else:
            employee_name = clean_value(
                row.get("Nama bawahan yang dinilai", row_dict.get("nama bawahan yang dinilai", ""))
            )
            superior_name = clean_value(row.get("Nama", row_dict.get("nama", "")))

        email = clean_value(row.get("Email", row_dict.get("email", "")))
        division = clean_value(row.get("Divisi", row_dict.get("divisi", "")))
        unit = clean_value(row.get("Unit", row_dict.get("unit", "")))
        department = clean_value(row.get("Departemen", row_dict.get("departemen", "")))
        directorate = clean_value(row.get("Direktorat", row_dict.get("direktorat", "")))
        current_level = clean_value(row.get("Level Saat Ini", row_dict.get("level saat ini", "")))
        target_level = clean_value(row.get("Level Tujuan", row_dict.get("level tujuan", "")))
        next_path = clean_value(row.get("Level Next Path", row_dict.get("level next path", "")))
        completion_time = row.get("Completion time", row_dict.get("completion time", None))
        last_modified_time = row.get("Last modified time", row_dict.get("last modified time", None))

        for question_col in question_columns:
            question_normalized = normalize_text(question_col)
            answer = row.get(question_col, "")

            if pd.isna(answer):
                continue

            answer_str = str(answer).strip()
            if not answer_str:
                continue

            answer_normalized = normalize_text(answer_str)

            master_rows = indicator_master_df[
                indicator_master_df["question_normalized"] == question_normalized
            ]

            if master_rows.empty:
                results.append({
                    "rater_type": rater_type,
                    "employee_name": employee_name,
                    "superior_name": superior_name,
                    "email": email,
                    "division": division,
                    "unit": unit,
                    "department": department,
                    "directorate": directorate,
                    "current_level": current_level,
                    "target_level": target_level,
                    "next_path": next_path,
                    "completion_time": completion_time,
                    "last_modified_time": last_modified_time,
                    "question": question_col,
                    "answer_text": answer_str,
                    "rating": np.nan,
                    "mapping_status": "QUESTION_NOT_FOUND",
                    "indicator_id": "",
                    "dimension": "",
                    "indicator_no": np.nan,
                    "indicator": "",
                })
                continue

            master_row = master_rows.iloc[0]
            rating = answer_mapping[question_normalized].get(answer_normalized, np.nan)
            mapping_status = "ANSWER_NOT_FOUND" if pd.isna(rating) else "MATCHED"

            results.append({
                "rater_type": rater_type,
                "employee_name": employee_name,
                "superior_name": superior_name,
                "email": email,
                "division": division,
                "unit": unit,
                "department": department,
                "directorate": directorate,
                "current_level": current_level,
                "target_level": target_level,
                "next_path": next_path,
                "completion_time": completion_time,
                "last_modified_time": last_modified_time,
                "question": question_col,
                "answer_text": answer_str,
                "rating": rating,
                "mapping_status": mapping_status,
                "indicator_id": master_row["indicator_id"],
                "dimension": master_row["dimension"],
                "indicator_no": master_row["indicator_no"],
                "indicator": master_row["indicator"],
            })

    return pd.DataFrame(results)

In [ ]:
# ============================================================
# CELL 10 — PROCESS SELF
# ============================================================

self_scored_df = process_assessment(
    df=df_self,
    question_columns=self_question_cols,
    rater_type="Self"
)

print("========================================")
print("SELF ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(self_scored_df)
)

print(
    "\nMapping status:"
)

print(
    self_scored_df[
        "mapping_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# CELL 11 — PROCESS SUPERIOR
# ============================================================

superior_scored_df = process_assessment(
    df=df_superior,
    question_columns=superior_question_cols,
    rater_type="Superior"
)

print("========================================")
print("SUPERIOR ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(superior_scored_df)
)

print(
    "\nMapping status:"
)

print(
    superior_scored_df[
        "mapping_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# CELL 12 — COMBINE + VALIDATE SCORING
# ============================================================

scored_answers_df = pd.concat(
    [
        self_scored_df,
        superior_scored_df
    ],
    ignore_index=True
)

total_records = len(
    scored_answers_df
)

matched_records = (
    scored_answers_df[
        scored_answers_df["mapping_status"] == "MATCHED"
    ].shape[0]
)

unmatched_records = (
    total_records - matched_records
)

coverage = (
    matched_records / total_records
    if total_records > 0
    else 0
)

print("========================================")
print("SCORING VALIDATION")
print("========================================")

print("Total records :", total_records)
print("Matched       :", matched_records)
print("Unmatched     :", unmatched_records)
print("Coverage      :", f"{coverage:.2%}")

# ------------------------------------------------------------
# EMPLOYEE VALIDATION
# ------------------------------------------------------------

empty_employee = (
    scored_answers_df["employee_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty employee:",
    empty_employee
)

if unmatched_records > 0:

    print(
        "\nWARNING: Ada jawaban yang tidak cocok."
    )

    display(
        scored_answers_df[
            scored_answers_df[
                "mapping_status"
            ] != "MATCHED"
        ][
            [
                "rater_type",
                "employee_name",
                "question",
                "answer_text",
                "mapping_status"
            ]
        ].head(50)
    )

else:

    print(
        "\n100% answer mapping berhasil."
    )

assert empty_employee == 0, (
    "Ada record dengan employee_name kosong."
)

In [ ]:
# ============================================================
# CELL 13 — INDICATOR SCORE
# ============================================================

indicator_score_df = (
    scored_answers_df[
        scored_answers_df[
            "mapping_status"
        ] == "MATCHED"
    ]
    .groupby(
        [
            "rater_type",
            "employee_name",
            "indicator_id"
        ],
        as_index=False
    )
    ["rating"]
    .mean()
    .rename(
        columns={
            "rating": "indicator_score"
        }
    )
)

indicator_score_df = (
    indicator_score_df.merge(
        indicator_master_df[
            [
                "indicator_id",
                "dimension",
                "indicator_no",
                "indicator"
            ]
        ],
        on="indicator_id",
        how="left"
    )
)

print("========================================")
print("INDICATOR SCORE")
print("========================================")

print(
    indicator_score_df.head(20)
)

In [ ]:
# ============================================================
# CELL 14 — 5 DIMENSION SCORE
# ============================================================

dimension_score_df = (
    indicator_score_df
    .groupby(
        [
            "rater_type",
            "employee_name",
            "dimension"
        ],
        as_index=False
    )
    ["indicator_score"]
    .mean()
    .rename(
        columns={
            "indicator_score":
                "dimension_score"
        }
    )
)

print("========================================")
print("DIMENSION SCORE")
print("========================================")

display(
    dimension_score_df
)

In [ ]:
# ============================================================
# CELL 15 — SELF VS SUPERIOR GAP
# ============================================================

dimension_gap_df = (
    dimension_score_df
    .pivot_table(
        index=[
            "employee_name",
            "dimension"
        ],
        columns="rater_type",
        values="dimension_score"
    )
    .reset_index()
)

# ------------------------------------------------------------
# ENSURE COLUMNS
# ------------------------------------------------------------

if "Self" not in dimension_gap_df.columns:
    dimension_gap_df["Self"] = np.nan

if "Superior" not in dimension_gap_df.columns:
    dimension_gap_df["Superior"] = np.nan

# ------------------------------------------------------------
# GAP
# ------------------------------------------------------------

dimension_gap_df["gap"] = (
    dimension_gap_df["Self"]
    -
    dimension_gap_df["Superior"]
)

print("========================================")
print("DIMENSION GAP")
print("========================================")

display(
    dimension_gap_df
)

In [ ]:
# CELL 16 — MASTER INPUT CELL CONFIGURATION

IDENTITY_INPUT_CELLS = [
    "D2",   # Nama
    "D3",   # Nama Atasan
    "D4",   # Level Saat Ini
    "D5",   # Level Tujuan
    "D6",   # Tanggal Assessment
    "D7",   # Unit
    "D8",   # Departemen
    "D9",   # Divisi
    "D10",  # Direktorat
    "D12",  # Level Next Path
]

# ============================================================
# ASSESSMENT INPUT
# ============================================================

ASSESSMENT_INPUT_COLUMNS = {

    # I = Self Rating
    "self_rating": 9,

    # J = Self Answer / Comment
    "self_comment": 10,

    # K = Superior Rating
    "superior_rating": 11,

    # L = Superior Answer / Comment
    "superior_comment": 12,
}


# ============================================================
# ROW INDICATOR
# ============================================================

ASSESSMENT_ROWS = list(
    range(13, 23)
)


# ============================================================
# DIMENSION SHEETS
# ============================================================

ASSESSMENT_SHEETS = list(
    DIMENSIONS.values()
)


print("========================================")
print("MASTER INPUT CONFIGURATION")
print("========================================")

print(
    "Identity input cells :",
    IDENTITY_INPUT_CELLS
)

print(
    "Assessment sheets    :",
    ASSESSMENT_SHEETS
)

print(
    "Assessment rows      :",
    f"{ASSESSMENT_ROWS[0]}-{ASSESSMENT_ROWS[-1]}"
)

print()
print("Python hanya akan mengisi input cells.")
print("Formula dan formatting mengikuti master.")

In [ ]:
# ============================================================
# CELL 17 — FILL MASTER FOR ONE EMPLOYEE
# ============================================================

from datetime import datetime, date


# ============================================================
# HELPER 1 — CLEAN EXCEL VALUE
# ============================================================

def clean_excel_value(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return None

    return value


# ============================================================
# HELPER 2 — FIND COLUMN
# ============================================================

def find_column(df, candidates):

    if df is None or df.empty:
        return None

    normalized_columns = {
        normalize_text(col): col
        for col in df.columns
    }

    for candidate in candidates:

        candidate_normalized = normalize_text(
            candidate
        )

        if candidate_normalized in normalized_columns:
            return normalized_columns[
                candidate_normalized
            ]

    return None


# ============================================================
# HELPER 3 — WRITE VALUE ONLY
# ============================================================

def write_value_only(cell, value):

    """
    Menulis VALUE saja.

    Tidak mengubah:
    - font
    - fill
    - border
    - alignment
    - number format
    - protection
    """

    value = clean_excel_value(value)

    if value is None:
        cell.value = None

    else:
        cell.value = value


# ============================================================
# MAIN FUNCTION
# ============================================================

def fill_master_for_employee(
    employee_name,
    master_file,
    scored_answers
):

    # ========================================================
    # 1. LOAD MASTER TEMPLATE
    # ========================================================

    wb = load_workbook(
        master_file,
        data_only=False
    )


    # ========================================================
    # 2. TARGET EMPLOYEE
    # ========================================================

    target_employee = normalize_text(
        employee_name
    )


    # ========================================================
    # 3. FILTER EMPLOYEE DATA
    # ========================================================

    employee_data = scored_answers[
        scored_answers["employee_name"]
        .apply(normalize_text)
        == target_employee
    ].copy()


    if employee_data.empty:

        raise ValueError(
            f"Tidak ada data assessment untuk employee: "
            f"{employee_name}"
        )


    # ========================================================
    # 4. SHEET IDENTITAS
    # ========================================================

    ws_identity = wb["Identitas"]


    # ========================================================
    # D2 — NAMA
    # ========================================================

    write_value_only(
        ws_identity["D2"],
        employee_name
    )


    # ========================================================
    # 5. CARI DATA SELF RAW
    # ========================================================

    self_raw = pd.DataFrame()

    if df_self is not None and not df_self.empty:

        self_name_col = find_column(
            df_self,
            [
                "Nama",
                "Name"
            ]
        )

        if self_name_col is not None:

            self_raw = df_self[
                df_self[self_name_col]
                .apply(normalize_text)
                == target_employee
            ].copy()


    # ========================================================
    # 6. CARI NAMA ATASAN
    # ========================================================

    superior_name = None

    if df_superior is not None and not df_superior.empty:

        subordinate_col = find_column(
            df_superior,
            [
                "Nama bawahan yang dinilai",
                "Nama Bawahan",
                "Bawahan"
            ]
        )

        if subordinate_col is not None:

            superior_employee_data = df_superior[
                df_superior[subordinate_col]
                .apply(normalize_text)
                == target_employee
            ].copy()


            if not superior_employee_data.empty:

                superior_name_col = find_column(
                    df_superior,
                    [
                        "Nama",
                        "Name",
                        "Nama atasan yang menilai",
                        "Nama atasan",
                        "Nama superior",
                        "Superior Name"
                    ]
                )


                if superior_name_col is not None:

                    superior_name = clean_excel_value(
                        superior_employee_data.iloc[0][
                            superior_name_col
                        ]
                    )


    # ========================================================
    # D3 — NAMA ATASAN
    # ========================================================

    write_value_only(
        ws_identity["D3"],
        superior_name
    )


    # ========================================================
    # 7. IDENTITY MAPPING
    # ========================================================

    identity_mapping = {

        "D4": [
            "Level Saat Ini",
            "Level saat ini",
            "Current Level"
        ],

        "D5": [
            "Level Tujuan",
            "Level tujuan",
            "Target Level"
        ],

        "D6": [
            "Tanggal Assesment",
            "Tanggal Assessment",
            "Assessment Date",
            "Completion time",
            "Start time",
            "Last modified time"
        ],

        "D7": [
            "Unit"
        ],

        "D8": [
            "Departemen",
            "Department"
        ],

        "D9": [
            "Divisi",
            "Division"
        ],

        "D10": [
            "Direktorat",
            "Directorate"
        ],

        "D12": [
            "Level Next Path",
            "Next Path",
            "Level Next"
        ]
    }


    # ========================================================
    # 8. ISI IDENTITY DARI SELF RAW
    # ========================================================

    if not self_raw.empty:

        self_row = self_raw.iloc[0]

        for target_cell, candidates in identity_mapping.items():

            source_col = find_column(
                self_raw,
                candidates
            )

            if source_col is None:
                continue


            value = clean_excel_value(
                self_row[source_col]
            )


            if value is None:
                continue


            # ------------------------------------------------
            # D6 — TANGGAL
            # ------------------------------------------------

            if target_cell == "D6":

                if isinstance(value, pd.Timestamp):

                    value = value.to_pydatetime()


                # Jangan mengatur number_format.
                #
                # Format tanggal mengikuti master.

                write_value_only(
                    ws_identity[target_cell],
                    value
                )


            else:

                write_value_only(
                    ws_identity[target_cell],
                    value
                )


    # ========================================================
    # 9. PROCESS 5 DIMENSIONS
    # ========================================================

    for dimension_key, sheet_name in DIMENSIONS.items():

        ws = wb[sheet_name]


        # ----------------------------------------------------
        # FILTER EMPLOYEE + DIMENSION
        # ----------------------------------------------------

        dimension_data = employee_data[
            employee_data["dimension"]
            == dimension_key
        ].copy()


        # ====================================================
        # LOOP 10 INDICATORS
        # ====================================================

        for indicator_no, row in enumerate(
            ASSESSMENT_ROWS,
            start=1
        ):

            indicator_id = (
                f"{dimension_key}_{indicator_no:02d}"
            )


            # ------------------------------------------------
            # DATA INDICATOR
            # ------------------------------------------------

            indicator_data = dimension_data[
                dimension_data["indicator_id"]
                == indicator_id
            ].copy()


            # =================================================
            # SELF
            # =================================================

            self_data = indicator_data[
                indicator_data["rater_type"]
                == "Self"
            ].copy()


            self_rating = None

            if not self_data.empty:

                ratings = pd.to_numeric(
                    self_data["rating"],
                    errors="coerce"
                ).dropna()

                if not ratings.empty:

                    self_rating = ratings.iloc[0]


            # ------------------------------------------------
            # I = SELF RATING
            # ------------------------------------------------

            write_value_only(
                ws.cell(
                    row=row,
                    column=ASSESSMENT_INPUT_COLUMNS[
                        "self_rating"
                    ]
                ),
                self_rating
            )


            # ------------------------------------------------
            # J = SELF ANSWER
            # ------------------------------------------------

            if not self_data.empty:

                self_answers = (
                    self_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )

                self_answers = self_answers[
                    self_answers != ""
                ]


                if not self_answers.empty:

                    self_comment = "\n".join(
                        self_answers.tolist()
                    )

                else:

                    self_comment = None

            else:

                self_comment = None


            write_value_only(
                ws.cell(
                    row=row,
                    column=ASSESSMENT_INPUT_COLUMNS[
                        "self_comment"
                    ]
                ),
                self_comment
            )


            # =================================================
            # SUPERIOR
            # =================================================

            superior_data = indicator_data[
                indicator_data["rater_type"]
                == "Superior"
            ].copy()


            superior_rating = None

            if not superior_data.empty:

                ratings = pd.to_numeric(
                    superior_data["rating"],
                    errors="coerce"
                ).dropna()

                if not ratings.empty:

                    superior_rating = ratings.iloc[0]


            # ------------------------------------------------
            # K = SUPERIOR RATING
            # ------------------------------------------------

            write_value_only(
                ws.cell(
                    row=row,
                    column=ASSESSMENT_INPUT_COLUMNS[
                        "superior_rating"
                    ]
                ),
                superior_rating
            )


            # ------------------------------------------------
            # L = SUPERIOR ANSWER
            # ------------------------------------------------

            if not superior_data.empty:

                superior_answers = (
                    superior_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )

                superior_answers = superior_answers[
                    superior_answers != ""
                ]


                if not superior_answers.empty:

                    superior_comment = "\n".join(
                        superior_answers.tolist()
                    )

                else:

                    superior_comment = None

            else:

                superior_comment = None


            write_value_only(
                ws.cell(
                    row=row,
                    column=ASSESSMENT_INPUT_COLUMNS[
                        "superior_comment"
                    ]
                ),
                superior_comment
            )

    return wb

In [ ]:
# ============================================================
# CELL 17B — EXTRACT INPUT VALUES
# ============================================================

def extract_master_input_values(wb):

    input_values = {}

    # ========================================================
    # IDENTITAS
    # ========================================================

    for cell_address in IDENTITY_INPUT_CELLS:

        input_values[
            ("Identitas", cell_address)
        ] = wb["Identitas"][cell_address].value


    # ========================================================
    # 5 DIMENSIONS
    # ========================================================

    for sheet_name in ASSESSMENT_SHEETS:

        ws = wb[sheet_name]

        for row in ASSESSMENT_ROWS:

            for column_number in ASSESSMENT_INPUT_COLUMNS.values():

                cell = ws.cell(
                    row=row,
                    column=column_number
                )

                input_values[
                    (
                        sheet_name,
                        cell.coordinate
                    )
                ] = cell.value


    return input_values

In [ ]:
# ============================================================
# CELL 17C — SAVE USING MICROSOFT EXCEL
# ============================================================

def save_using_excel(
    master_file,
    output_file,
    input_values
):

    master_file = Path(master_file)
    output_file = Path(output_file)


    # ========================================================
    # 1. COPY MASTER ASLI
    # ========================================================

    if output_file.exists():

        try:
            output_file.unlink()

        except PermissionError:

            raise PermissionError(
                f"Output file sedang digunakan oleh aplikasi lain:\n"
                f"{output_file}\n\n"
                f"Tutup file Excel tersebut lalu jalankan kembali."
            )


    shutil.copy2(
        master_file,
        output_file
    )


    # ========================================================
    # 2. OPEN MICROSOFT EXCEL
    # ========================================================

    excel = win32.DispatchEx(
        "Excel.Application"
    )

    excel.Visible = False
    excel.DisplayAlerts = False
    excel.ScreenUpdating = False

    # Manual calculation
    excel.Calculation = -4135


    excel_wb = None


    try:

        # ====================================================
        # 3. OPEN OUTPUT FILE
        # ====================================================

        excel_wb = excel.Workbooks.Open(
            str(output_file),
            UpdateLinks=0,
            ReadOnly=False,
            AddToMru=False
        )


        # ====================================================
        # 4. WRITE VALUES ONLY
        # ====================================================

        for (
            sheet_name,
            cell_address
        ), value in input_values.items():

            ws = excel_wb.Worksheets(
                sheet_name
            )

            cell = ws.Range(
                cell_address
            )


            # ------------------------------------------------
            # EMPTY VALUE
            # ------------------------------------------------

            if value is None:

                cell.ClearContents()

                continue


            # ------------------------------------------------
            # PANDAS TIMESTAMP
            # ------------------------------------------------

            if isinstance(
                value,
                pd.Timestamp
            ):

                value = value.to_pydatetime()


            # ------------------------------------------------
            # NUMPY VALUE
            # ------------------------------------------------

            if isinstance(
                value,
                np.generic
            ):

                value = value.item()


            # ------------------------------------------------
            # WRITE VALUE ONLY
            # ------------------------------------------------

            cell.Value = value


        # ====================================================
        # 5. SAVE
        # ====================================================

        excel_wb.Save()


    finally:

        # ====================================================
        # CLOSE WORKBOOK
        # ====================================================

        if excel_wb is not None:

            try:

                excel_wb.Close(
                    SaveChanges=False
                )

            except Exception:

                pass


        # ====================================================
        # QUIT EXCEL
        # ====================================================

        try:

            excel.Quit()

        except Exception:

            pass

In [ ]:
# ============================================================
# CELL 18 — PRE-GENERATION VALIDATION
# ============================================================

print("=" * 60)
print("PRE-GENERATION VALIDATION")
print("=" * 60)


# ============================================================
# VALIDATE DIMENSIONS
# ============================================================

assert len(DIMENSIONS) == 5, (
    "Jumlah dimension harus 5."
)


# ============================================================
# VALIDATE INDICATORS
# ============================================================

assert len(ASSESSMENT_ROWS) == 10, (
    "Setiap dimension harus memiliki 10 indicator."
)


# ============================================================
# VALIDATE SCORED DATA
# ============================================================

required_columns = [
    "rater_type",
    "employee_name",
    "indicator_id",
    "dimension",
    "rating",
    "answer_text",
]

missing_columns = [
    col
    for col in required_columns
    if col not in scored_answers_df.columns
]

assert not missing_columns, (
    f"Kolom scored_answers_df missing: "
    f"{missing_columns}"
)


# ============================================================
# VALIDATE EMPLOYEE
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)


assert len(employees) > 0, (
    "Tidak ada employee Self Assessment."
)


print("Dimensions :", len(DIMENSIONS))
print("Indicators :", len(ASSESSMENT_ROWS))
print("Employees  :", len(employees))

print()
print("Pre-generation validation: OK")

In [ ]:
# ============================================================
# CELL 19 — MASTER STRUCTURE SNAPSHOT
# ============================================================

def get_master_structure(master_file):

    wb = load_workbook(
        master_file,
        data_only=False
    )

    structure = {}

    for sheet_name in wb.sheetnames:

        ws = wb[sheet_name]

        structure[sheet_name] = {

            "max_row": ws.max_row,

            "max_column": ws.max_column,

            "merged_ranges": tuple(
                sorted(
                    str(rng)
                    for rng in ws.merged_cells.ranges
                )
            ),

            "column_dimensions": {
                key: (
                    dim.width,
                    dim.hidden
                )
                for key, dim
                in ws.column_dimensions.items()
            },

            "row_dimensions": {
                key: (
                    dim.height,
                    dim.hidden
                )
                for key, dim
                in ws.row_dimensions.items()
            },
        }

    return structure


MASTER_STRUCTURE = get_master_structure(
    MASTER_FILE
)


print("=" * 60)
print("MASTER STRUCTURE SNAPSHOT")
print("=" * 60)

for sheet_name, info in MASTER_STRUCTURE.items():

    print(
        f"{sheet_name:35} "
        f"{info['max_row']} rows × "
        f"{info['max_column']} cols"
    )

print()
print("Master structure snapshot created.")

In [ ]:
# ============================================================
# CELL 20 — OUTPUT STRUCTURE VALIDATION
# ============================================================

def validate_output_structure(
    output_file,
    master_structure
):

    wb = load_workbook(
        output_file,
        data_only=False
    )


    # ========================================================
    # SHEET CHECK
    # ========================================================

    if wb.sheetnames != list(
        master_structure.keys()
    ):

        return {
            "status": "FAILED",
            "reason": "Sheet structure changed."
        }


    # ========================================================
    # STRUCTURE CHECK
    # ========================================================

    for sheet_name, master_info in master_structure.items():

        ws = wb[sheet_name]


        # ----------------------------------------------------
        # DIMENSION
        # ----------------------------------------------------

        if ws.max_row != master_info["max_row"]:

            return {
                "status": "FAILED",
                "reason": (
                    f"{sheet_name}: max_row changed."
                )
            }


        if ws.max_column != master_info["max_column"]:

            return {
                "status": "FAILED",
                "reason": (
                    f"{sheet_name}: max_column changed."
                )
            }


        # ----------------------------------------------------
        # MERGED CELLS
        # ----------------------------------------------------

        output_merged = tuple(
            sorted(
                str(rng)
                for rng in ws.merged_cells.ranges
            )
        )

        if output_merged != master_info[
            "merged_ranges"
        ]:

            return {
                "status": "FAILED",
                "reason": (
                    f"{sheet_name}: merged cells changed."
                )
            }


    return {
        "status": "PASSED",
        "reason": "Structure preserved."
    }

In [ ]:
# ============================================================
# CELL 20B — INPUT FONT SNAPSHOT
# ============================================================

def get_input_font_snapshot(file_path):

    wb = load_workbook(
        file_path,
        data_only=False
    )

    result = {}


    # ========================================================
    # IDENTITAS
    # ========================================================

    ws = wb["Identitas"]

    for cell_address in IDENTITY_INPUT_CELLS:

        cell = ws[cell_address]

        result[
            ("Identitas", cell_address)
        ] = {

            "font_name": cell.font.name,

            "font_size": cell.font.sz,

            "bold": cell.font.bold,

            "italic": cell.font.italic,

            "underline": cell.font.underline,

            "strike": cell.font.strike,

        }


    # ========================================================
    # 5 DIMENSIONS
    # ========================================================

    for sheet_name in ASSESSMENT_SHEETS:

        ws = wb[sheet_name]

        for row in ASSESSMENT_ROWS:

            for column_number in (
                ASSESSMENT_INPUT_COLUMNS.values()
            ):

                cell = ws.cell(
                    row=row,
                    column=column_number
                )

                result[
                    (
                        sheet_name,
                        cell.coordinate
                    )
                ] = {

                    "font_name": cell.font.name,

                    "font_size": cell.font.sz,

                    "bold": cell.font.bold,

                    "italic": cell.font.italic,

                    "underline": cell.font.underline,

                    "strike": cell.font.strike,

                }


    wb.close()

    return result


# ============================================================
# CREATE MASTER FONT SNAPSHOT
# ============================================================

MASTER_INPUT_FONTS = get_input_font_snapshot(
    MASTER_FILE
)


print("=" * 60)
print("INPUT FONT SNAPSHOT")
print("=" * 60)

print(
    "Total input cells monitored:",
    len(MASTER_INPUT_FONTS)
)

print()

# Sample
for key, font_info in list(
    MASTER_INPUT_FONTS.items()
)[:10]:

    print(
        f"{key[0]}!{key[1]} → "
        f"{font_info['font_name']} "
        f"{font_info['font_size']}pt"
    )

print()
print("Input font snapshot created.")

In [ ]:
# ============================================================
# CELL 20C — VALIDATE INPUT FONT PRESERVATION
# ============================================================

def compare_input_fonts(
    output_file,
    master_fonts
):

    output_fonts = get_input_font_snapshot(
        output_file
    )

    differences = []


    # ========================================================
    # CHECK CELL SET
    # ========================================================

    if set(output_fonts.keys()) != set(
        master_fonts.keys()
    ):

        return [
            {
                "sheet": "SYSTEM",
                "cell": "SYSTEM",
                "master": None,
                "output": None,
                "reason": (
                    "Jumlah / daftar input cell berubah."
                )
            }
        ]


    # ========================================================
    # COMPARE
    # ========================================================

    for key, master_font in master_fonts.items():

        output_font = output_fonts.get(
            key
        )

        if output_font != master_font:

            differences.append({

                "sheet": key[0],

                "cell": key[1],

                "master": master_font,

                "output": output_font,

            })


    return differences


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate_input_font_preservation(
    output_file,
    master_fonts
):

    differences = compare_input_fonts(
        output_file,
        master_fonts
    )


    if differences:

        return {

            "status": "FAILED",

            "reason": (
                f"Ditemukan {len(differences)} "
                f"input cell dengan font berbeda."
            ),

            "differences": differences

        }


    return {

        "status": "PASSED",

        "reason": (
            "Semua font input cell "
            "sesuai dengan master."
        ),

        "differences": []

    }


print("=" * 60)
print("FONT VALIDATION FUNCTION READY")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 21 — FORMULA PRESERVATION CHECK
# ============================================================

def get_formula_snapshot(file_path):

    wb = load_workbook(
        file_path,
        data_only=False
    )

    formulas = {}


    for sheet_name in wb.sheetnames:

        ws = wb[sheet_name]

        sheet_formulas = {}


        for row in ws.iter_rows():

            for cell in row:

                if (
                    isinstance(cell.value, str)
                    and cell.value.startswith("=")
                ):

                    sheet_formulas[
                        cell.coordinate
                    ] = cell.value


        formulas[
            sheet_name
        ] = sheet_formulas


    wb.close()

    return formulas


# ============================================================
# NORMALIZE FORMULA
# ============================================================

def normalize_formula(formula):

    if not isinstance(
        formula,
        str
    ):

        return formula


    # --------------------------------------------------------
    # REMOVE LINE BREAK
    # --------------------------------------------------------

    formula = formula.replace(
        "\n",
        ""
    )

    formula = formula.replace(
        "\r",
        ""
    )


    # --------------------------------------------------------
    # REMOVE ALL WHITESPACE
    # --------------------------------------------------------

    formula = re.sub(
        r"\s+",
        "",
        formula
    )


    return formula


# ============================================================
# MASTER FORMULA SNAPSHOT
# ============================================================

MASTER_FORMULAS = get_formula_snapshot(
    MASTER_FILE
)


# ============================================================
# VALIDATE FORMULA
# ============================================================

def validate_formula_preservation(
    output_file,
    master_formulas
):

    output_formulas = get_formula_snapshot(
        output_file
    )


    # ========================================================
    # SHEET CHECK
    # ========================================================

    if set(output_formulas.keys()) != set(
        master_formulas.keys()
    ):

        return {

            "status": "FAILED",

            "reason": (
                "Formula sheet structure changed."
            )

        }


    # ========================================================
    # FORMULA COUNT CHECK
    # ========================================================

    for sheet_name in master_formulas:

        master_count = len(
            master_formulas[sheet_name]
        )

        output_count = len(
            output_formulas[sheet_name]
        )

        if master_count != output_count:

            return {

                "status": "FAILED",

                "reason": (
                    f"{sheet_name}: "
                    f"formula count changed. "
                    f"Master={master_count}, "
                    f"Output={output_count}"
                )

            }


    # ========================================================
    # FORMULA CONTENT CHECK
    # ========================================================

    differences = []


    for sheet_name in master_formulas:

        master_sheet = master_formulas[
            sheet_name
        ]

        output_sheet = output_formulas[
            sheet_name
        ]


        for coordinate, master_formula in (
            master_sheet.items()
        ):

            output_formula = output_sheet.get(
                coordinate
            )


            # ------------------------------------------------
            # FORMULA MISSING
            # ------------------------------------------------

            if output_formula is None:

                differences.append({

                    "sheet": sheet_name,

                    "cell": coordinate,

                    "master": master_formula,

                    "output": None,

                })

                continue


            # ------------------------------------------------
            # NORMALIZED COMPARISON
            # ------------------------------------------------

            master_normalized = normalize_formula(
                master_formula
            )

            output_normalized = normalize_formula(
                output_formula
            )


            if (
                master_normalized
                !=
                output_normalized
            ):

                differences.append({

                    "sheet": sheet_name,

                    "cell": coordinate,

                    "master": master_formula,

                    "output": output_formula,

                })


    # ========================================================
    # RESULT
    # ========================================================

    if differences:

        return {

            "status": "FAILED",

            "reason": (
                f"Ditemukan {len(differences)} "
                f"formula yang berbeda."
            ),

            "differences": differences

        }


    return {

        "status": "PASSED",

        "reason": (
            "All master formulas preserved."
        ),

        "differences": []

    }


print("=" * 60)
print("FORMULA SNAPSHOT CREATED")
print("=" * 60)

total_formulas = sum(
    len(v)
    for v in MASTER_FORMULAS.values()
)

print(
    "Total master formulas:",
    total_formulas
)

print(
    "Formula validator ready."
)

In [ ]:
# ============================================================
# CELL 22 — OUTPUT INPUT VALIDATION
# ============================================================

def validate_employee_output(
    output_file,
    employee_name
):

    wb = load_workbook(
        output_file,
        data_only=False
    )

    ws_identity = wb["Identitas"]

    errors = []


    # ========================================================
    # CHECK NAME
    # ========================================================

    output_name = ws_identity["D2"].value

    if normalize_text(output_name) != normalize_text(
        employee_name
    ):

        errors.append(
            "Identitas!D2 tidak sesuai employee."
        )


    # ========================================================
    # CHECK 5 DIMENSIONS
    # ========================================================

    for sheet_name in ASSESSMENT_SHEETS:

        if sheet_name not in wb.sheetnames:

            errors.append(
                f"Sheet {sheet_name} tidak ditemukan."
            )


    if errors:

        return {
            "status": "FAILED",
            "errors": errors
        }


    return {
        "status": "PASSED",
        "errors": []
    }

In [ ]:
# ============================================================
# CELL 25 — GENERATE MASTER UNTUK SEMUA KARYAWAN
# ============================================================

import gc
import time

print("=" * 70)
print("MASTER GENERATION — SEMUA KARYAWAN")
print("=" * 70)


# ============================================================
# 1. PREPARE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. RESULT CONTAINERS
# ============================================================

generated_files = []
failed_files = []


# ============================================================
# 3. GET EMPLOYEE LIST
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[
        lambda s: s != ""
    ]
    .unique()
)


print()
print(
    "Total employee:",
    len(employees)
)

print()


# ============================================================
# 4. VALIDATION CONFIGURATION
# ============================================================

# IMPORTANT:
#
# 1 = SEMUA employee divalidasi
#
# Validation yang dilakukan:
# - Workbook structure
# - Formula preservation
# - Employee data
# - Input font preservation
#
# Kita sengaja menggunakan 1 untuk debugging.
# Setelah pipeline sudah stabil, interval bisa dipertimbangkan
# untuk dipercepat.

VALIDATION_INTERVAL = 1


print(
    "Validation interval:",
    VALIDATION_INTERVAL
)

print(
    "Semua employee akan divalidasi."
)

print()


# ============================================================
# 5. START TIMER
# ============================================================

generation_start = time.time()


# ============================================================
# 6. GENERATE EMPLOYEE ONE BY ONE
# ============================================================

for index, employee_name in enumerate(
    employees,
    start=1
):

    employee_start = time.time()

    output_file = None

    wb = None

    print()
    print("=" * 70)
    print(
        f"[{index}/{len(employees)}] "
        f"Processing: {employee_name}"
    )
    print("=" * 70)


    try:

        # ====================================================
        # 6.1 SAFE FILE NAME
        # ====================================================

        safe_name = re.sub(
            r'[\\/*?:"<>|]',
            "_",
            str(employee_name)
        ).strip()


        output_file = (
            OUTPUT_DIR
            /
            f"Learning_Agility_{safe_name}.xlsx"
        )


        print(
            "    Output:",
            output_file.name
        )


        # ====================================================
        # 6.2 CREATE WORKBOOK IN MEMORY
        # ====================================================

        print(
            "    1. Creating workbook in memory..."
        )

        wb = fill_master_for_employee(

            employee_name=employee_name,

            master_file=MASTER_FILE,

            scored_answers=scored_answers_df

        )


        # ====================================================
        # 6.3 EXTRACT INPUT VALUES ONLY
        # ====================================================

        print(
            "    2. Extracting input values..."
        )

        input_values = (
            extract_master_input_values(
                wb
            )
        )


        # ====================================================
        # 6.4 CLOSE OPENPYXL WORKBOOK
        # ====================================================

        wb.close()

        wb = None

        gc.collect()


        # ====================================================
        # 6.5 COPY MASTER + WRITE VALUES USING EXCEL
        # ====================================================

        print(
            "    3. Copying master and writing values..."
        )

        save_using_excel(

            master_file=MASTER_FILE,

            output_file=output_file,

            input_values=input_values

        )


        # ====================================================
        # 6.6 VERIFY OUTPUT FILE EXISTS
        # ====================================================

        print(
            "    4. Checking output file..."
        )

        if not output_file.exists():

            raise FileNotFoundError(
                "Output file tidak berhasil dibuat:\n"
                f"{output_file}"
            )


        # Pastikan file tidak kosong
        if output_file.stat().st_size == 0:

            raise IOError(
                "Output file berhasil dibuat tetapi "
                "ukuran file = 0 byte:\n"
                f"{output_file}"
            )


        print(
            "       ✓ Output file exists"
        )

        print(
            "       ✓ File size:",
            f"{output_file.stat().st_size:,}",
            "bytes"
        )


        # ====================================================
        # 6.7 VALIDATION
        # ====================================================

        should_validate = (
            VALIDATION_INTERVAL is not None
            and index % VALIDATION_INTERVAL == 0
        )


        if should_validate:

            print()
            print(
                "    ------------------------------------------"
            )
            print(
                "    VALIDATING OUTPUT"
            )
            print(
                "    ------------------------------------------"
            )


            # =================================================
            # VALIDATION 1 — STRUCTURE
            # =================================================

            print(
                "    5. Validating workbook structure..."
            )

            structure_result = (
                validate_output_structure(

                    output_file,

                    MASTER_STRUCTURE

                )
            )


            if (
                structure_result["status"]
                !=
                "PASSED"
            ):

                raise ValueError(

                    "Structure validation failed: "
                    +
                    structure_result["reason"]

                )


            print(
                "       ✓ Structure passed"
            )


            # =================================================
            # VALIDATION 2 — FORMULA
            # =================================================

            print(
                "    6. Validating formulas..."
            )

            formula_result = (
                validate_formula_preservation(

                    output_file,

                    MASTER_FORMULAS

                )
            )


            if (
                formula_result["status"]
                !=
                "PASSED"
            ):

                print()
                print(
                    "       ✗ FORMULA DIFFERENCE"
                )


                differences = (
                    formula_result.get(
                        "differences",
                        []
                    )
                )


                for diff in differences[:20]:

                    print(
                        f"       {diff['sheet']}!"
                        f"{diff['cell']}"
                    )

                    print(
                        "          MASTER :",
                        diff["master"]
                    )

                    print(
                        "          OUTPUT :",
                        diff["output"]
                    )


                raise ValueError(

                    "Formula validation failed: "
                    +
                    formula_result["reason"]

                )


            print(
                "       ✓ Formula passed"
            )


            # =================================================
            # VALIDATION 3 — EMPLOYEE DATA
            # =================================================

            print(
                "    7. Validating employee data..."
            )

            employee_result = (
                validate_employee_output(

                    output_file,

                    employee_name

                )
            )


            if (
                employee_result["status"]
                !=
                "PASSED"
            ):

                raise ValueError(

                    "Employee validation failed: "
                    +
                    str(
                        employee_result[
                            "errors"
                        ]
                    )

                )


            print(
                "       ✓ Employee data passed"
            )


            # =================================================
            # VALIDATION 4 — INPUT FONT
            # =================================================

            print(
                "    8. Validating input fonts..."
            )

            font_result = (
                validate_input_font_preservation(

                    output_file,

                    MASTER_INPUT_FONTS

                )
            )


            if (
                font_result["status"]
                !=
                "PASSED"
            ):

                print()
                print(
                    "       ✗ INPUT FONT DIFFERENCE"
                )


                differences = (
                    font_result.get(
                        "differences",
                        []
                    )
                )


                for diff in differences[:20]:

                    print(
                        f"       "
                        f"{diff['sheet']}!"
                        f"{diff['cell']}"
                    )

                    print(
                        "          MASTER :",
                        diff["master"]
                    )

                    print(
                        "          OUTPUT :",
                        diff["output"]
                    )


                raise ValueError(

                    "Input font validation failed: "
                    +
                    font_result["reason"]

                )


            print(
                "       ✓ Input fonts passed"
            )


            print()
            print(
                "    ✓ ALL VALIDATION PASSED"
            )


        else:

            print(
                "    - Validation skipped"
            )


        # ====================================================
        # 6.8 SUCCESS
        # ====================================================

        generated_files.append(
            output_file
        )


        employee_elapsed = (
            time.time()
            -
            employee_start
        )


        print()
        print(
            "    ✓ SUCCESS"
        )

        print(
            "    Processing time:",
            f"{employee_elapsed:.2f} sec"
        )


    except Exception as e:

        employee_elapsed = (
            time.time()
            -
            employee_start
        )


        # ----------------------------------------------------
        # CLOSE WORKBOOK IF SOMETHING FAILED
        # ----------------------------------------------------

        if wb is not None:

            try:
                wb.close()
            except Exception:
                pass

            wb = None


        gc.collect()


        # ----------------------------------------------------
        # STORE FAILURE
        # ----------------------------------------------------

        failed_files.append({

            "employee": employee_name,

            "output_file": (
                str(output_file)
                if output_file is not None
                else None
            ),

            "error": repr(e),

            "processing_time": (
                f"{employee_elapsed:.2f} sec"
            )

        })


        # ----------------------------------------------------
        # PRINT FAILURE
        # ----------------------------------------------------

        print()
        print(
            "    ✗ FAILED"
        )

        print(
            "    Error:",
            repr(e)
        )

        print(
            "    Processing time:",
            f"{employee_elapsed:.2f} sec"
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # JANGAN menghentikan employee berikutnya
        # ----------------------------------------------------

        continue


    finally:

        # ====================================================
        # CLEAN MEMORY
        # ====================================================

        wb = None

        gc.collect()


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

total_elapsed = (
    time.time()
    -
    generation_start
)


print()
print()
print("=" * 70)
print("GENERATION SELESAI")
print("=" * 70)

print(
    "Total employee :",
    len(employees)
)

print(
    "Berhasil       :",
    len(generated_files)
)

print(
    "Gagal          :",
    len(failed_files)
)

print(
    "Total waktu    :",
    f"{total_elapsed:.2f} sec"
)

print(
    "Output         :",
    OUTPUT_DIR
)


# ============================================================
# 8. GENERATED FILE REPORT
# ============================================================

if generated_files:

    print()
    print(
        "GENERATED FILES:"
    )

    for file in generated_files:

        print(
            "✓",
            file.name
        )


# ============================================================
# 9. FAILED EMPLOYEE REPORT
# ============================================================

if failed_files:

    print()
    print(
        "FAILED EMPLOYEES:"
    )

    for item in failed_files:

        print(
            "-",
            item["employee"]
        )

        print(
            "  Output:",
            item["output_file"]
        )

        print(
            "  Error :",
            item["error"]
        )

        print(
            "  Time  :",
            item["processing_time"]
        )

else:

    print()
    print(
        "✓ ALL EMPLOYEES GENERATED AND VALIDATED SUCCESSFULLY"
    )


# ============================================================
# 10. FINAL STATUS
# ============================================================

print()
print("=" * 70)

if len(failed_files) == 0:

    print(
        "FINAL STATUS: SUCCESS"
    )

else:

    print(
        "FINAL STATUS: PARTIAL / FAILED"
    )

    print(
        f"{len(failed_files)} employee(s) perlu diperiksa."
    )

print("=" * 70)